# NexusRAG quickstart

Install from the repository root with `python -m pip install -e ".[eval]"` and use that environment as the notebook kernel. Start Ollama and run `ollama pull llama3.2:3b` for the default model. Embedding models may download on first use.

Settings come from environment variables and `.env`; see the [configuration guide](../CONTRIBUTING.md#configuration).


In [ ]:
import os
from pathlib import Path

cwd = Path.cwd().resolve()
repo_root = next(
    (
        path
        for path in (cwd, *cwd.parents)
        if (path / "setup.cfg").is_file() and (path / "src/scinexusrag/pipeline.py").is_file()
    ),
    None,
)
if repo_root is None:
    raise RuntimeError("Start this notebook inside the NexusRAG repository.")
os.chdir(repo_root)

from scinexusrag.config import get_settings  # noqa: E402
from scinexusrag.pipeline import NexusRAG  # noqa: E402

print(f"Repository: {repo_root}")

## 1. Initialize

Documents and indexes default to `data/`. Set `DATA_DIR` and `LANCEDB_PATH` to use another collection.


In [ ]:
settings = get_settings()
rag = NexusRAG(settings)

print(f"LLM model: {settings.llm.model}")
print(f"Embedding model: {settings.embedding.model}")
print(f"Storage path: {settings.storage.lancedb_path}")

## 2. Ingest a document

Replace `your-paper.pdf` with a PDF, DOCX, UTF-8 text or Markdown file. Relative paths start at the repository root.


In [ ]:
document_path = Path("your-paper.pdf").expanduser()
if not document_path.is_file():
    raise FileNotFoundError("Set document_path to an existing PDF, DOCX, text or Markdown file.")
result = rag.ingest(document_path)

print(f"Document ID: {result.document_id}")
print(f"Chunks created: {result.chunk_count}")
print(f"Word count: {result.word_count}")
print(f"Success: {result.success}")
if not result.success:
    raise RuntimeError(result.error or "Document ingestion failed.")

## 3. Ask a question

Search combines dense and BM25 rankings. Low dense similarity can trigger one feedback search. Check the cited passages; confidence does not establish correctness.


In [ ]:
response = rag.query("What are the main findings of the study?")

print(f"Answer: {response.answer}\n")
print(f"Confidence score: {response.confidence:.2f}")
print(f"Processing time: {response.processing_time_ms:.0f}ms")
print(f"\nSources ({len(response.sources)}):")
for i, source in enumerate(response.sources, 1):
    print(f"  [{i}] {source.document_name} (p.{source.page_number}, score: {source.score:.3f})")
    print(f"      {source.content[:100]}...")

## 4. List documents


In [ ]:
docs = rag.list_documents()
for doc in docs:
    name = doc.get("original_filename") or doc.get("filename", "Unknown")
    print(f"  {doc['id'][:8]}... | {name} | {doc.get('word_count', 0)} words")

stats = rag.get_stats()
print(f"\nTotal documents: {stats.total_documents}")
print(f"Total chunks: {stats.total_chunks}")
print(f"LLM available: {stats.llm_available}")

## 5. Inspect evaluation results

The next cell reads committed SciFact results without loading models.

Run the packaged sample evaluation from the repository root:

```bash
python -m scinexusrag.eval --sample
```

Uncached models need downloads. Full runs require the pinned datasets and models; see [evaluation commands and conditions](../docs/evaluation.md). New results go to `outputs/generated/`, and reports go to `outputs/generated/paper/`. Review scores and provenance before publishing.

These evaluations measure retrieval and evidence-sentence detection, not generated-answer accuracy.


In [ ]:
import json

results_path = repo_root / "outputs/results/scifact_test.json"
res = json.loads(results_path.read_text())
print(f"SciFact ablation ({res['num_queries']} claims, {res['corpus_size']} docs)")
for name, scores in res["systems"].items():
    print(f"  {name:20s} nDCG@10 = {scores['means']['nDCG@10']:.3f}")